1. Data Ingestion & Cleaning

In [0]:
spark

In [0]:
# PART 1: DATA INGESTION & CLEANING
# NYC Yellow Taxi Dataset

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml.feature import StringIndexer


# 1. LOAD DATA INTO SPARK DATAFRAME

# Option A: If file is uploaded into DBFS/FileStore
file_path = "/Volumes/trip/trip/trip/yellow_tripdata_2025-01.csv"


df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

print("Schema:")
df.printSchema()

print("Row count:", df.count())
display(df.limit(10))

Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: double (nullable = true)
 |-- tpep_dropoff_datetime: double (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

Row count: 1048575


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
1,1.73569E12,1.73569E12,1,1.6,1,N,229,237,1,10.0,3.5,0.5,3.0,0.0,1,18.0,2.5,0.0
1,1.73569E12,1.73569E12,1,0.5,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1,12.12,2.5,0.0
1,1.73569E12,1.73569E12,1,0.6,1,N,141,141,1,5.1,3.5,0.5,2.0,0.0,1,12.1,2.5,0.0
2,1.73569E12,1.73569E12,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.0,0.0,1,9.7,0.0,0.0
2,1.73569E12,1.73569E12,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.0,0.0,1,8.3,0.0,0.0
2,1.73569E12,1.73569E12,2,2.63,1,N,239,68,2,19.1,1.0,0.5,0.0,0.0,1,24.1,2.5,0.0
1,1.73569E12,1.73569E12,0,0.4,1,N,170,170,1,4.4,3.5,0.5,2.35,0.0,1,11.75,2.5,0.0
1,1.73569E12,1.73569E12,0,1.6,1,N,234,148,1,12.1,3.5,0.5,2.0,0.0,1,19.1,2.5,0.0
1,1.73569E12,1.73569E12,0,2.8,1,N,148,170,1,19.1,3.5,0.5,3.0,0.0,1,27.1,2.5,0.0
2,1.73569E12,1.73569E12,1,1.71,1,N,237,262,2,11.4,1.0,0.5,0.0,0.0,1,16.4,2.5,0.0


In [0]:
original_count = df.count()
print("Original rows:", original_count)

Original rows: 1048575


In [0]:
from pyspark.sql import functions as F

# Check nulls
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
])
display(null_counts)

# Remove duplicates
df = df.dropDuplicates()
print("After removing duplicates:", df.count())

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


After removing duplicates: 1046208


In [0]:
df = df.withColumn(
    "pickup_datetime",
    F.to_timestamp(F.from_unixtime(F.col("tpep_pickup_datetime") / 1000))
).withColumn(
    "dropoff_datetime",
    F.to_timestamp(F.from_unixtime(F.col("tpep_dropoff_datetime") / 1000))
)

df = df.drop("tpep_pickup_datetime", "tpep_dropoff_datetime")

In [0]:
numeric_cols = [
    "passenger_count", "trip_distance", "fare_amount", "extra",
    "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge",
    "total_amount", "congestion_surcharge", "Airport_fee"
]

median_values = {}
for c in numeric_cols:
    median_values[c] = df.approxQuantile(c, [0.5], 0.01)[0]

df = df.fillna(median_values)
df = df.fillna({"store_and_fwd_flag": "Unknown"})

In [0]:
df = df.filter(F.col("passenger_count") > 0) \
       .filter(F.col("trip_distance") > 0) \
       .filter(F.col("fare_amount") > 0) \
       .filter(F.col("total_amount") > 0) \
       .filter(F.col("pickup_datetime").isNotNull()) \
       .filter(F.col("dropoff_datetime").isNotNull()) \
       .filter(F.col("dropoff_datetime") >= F.col("pickup_datetime"))

In [0]:
def remove_outliers_iqr(dataframe, column_name):
    q1, q3 = dataframe.approxQuantile(column_name, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return dataframe.filter((F.col(column_name) >= lower) & (F.col(column_name) <= upper))

for col in ["trip_distance", "fare_amount", "total_amount", "tip_amount"]:
    df = remove_outliers_iqr(df, col)

In [0]:
print("FINAL DATA SUMMARY")
print("Original rows:", original_count)
print("Final rows:", df.count())
print("Total rows removed:", original_count - df.count())

FINAL DATA SUMMARY
Original rows: 1048575
Final rows: 831891
Total rows removed: 216684


In [0]:
from pyspark.sql import functions as F

# Create features
df = df.withColumn("pickup_hour", F.hour("pickup_datetime")) \
       .withColumn("pickup_day", F.dayofweek("pickup_datetime")) \
       .withColumn(
           "trip_duration",
           (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60
       )

# SAFE FILTERING (THIS IS THE FIX)
df = df.filter(
    (F.col("trip_duration").isNotNull()) &      # remove nulls
    (F.col("trip_duration") >= 0) &             # allow 0 (safe)
    (F.col("trip_duration") <= 300)             # remove extreme trips (>5 hours)
)

In [0]:
print("FINAL DATA SUMMARY")
print("Original rows:", original_count)
print("Final rows:", df.count())
print("Total rows removed:", original_count - df.count())

FINAL DATA SUMMARY
Original rows: 1048575
Final rows: 831561
Total rows removed: 217014


In [0]:
# Step 1: Repartition
df = df.repartition(8)

# Step 2: Save as Delta
output_path = "/Volumes/trip/trip/trip/delta_yellow_tripdata_2025-01.csv"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("pickup_day") \
    .save(output_path)

# Step 3: Reload
df = spark.read.format("delta").load(output_path)

# Step 4: Create temp view
df.createOrReplaceTempView("taxi_trips_cleaned")

In [0]:
df.createOrReplaceTempView("taxi_trips_cleaned")

In [0]:


# Check summary after cleaning
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    AVG(trip_distance) AS avg_trip_distance,
    AVG(fare_amount) AS avg_fare_amount,
    AVG(total_amount) AS avg_total_amount,
    AVG(trip_duration) AS avg_trip_duration
FROM taxi_trips_cleaned
""").show()

# Check categorical distribution
spark.sql("""
    SELECT payment_type, COUNT(*) AS trip_count
    FROM taxi_trips_cleaned
    GROUP BY payment_type
    ORDER BY trip_count DESC
""").show()


+----------+------------------+------------------+------------------+-----------------+
|total_rows| avg_trip_distance|   avg_fare_amount|  avg_total_amount|avg_trip_duration|
+----------+------------------+------------------+------------------+-----------------+
|    831561|1.6843300611741057|12.058040757082166|19.571393499695144|10.95529973146887|
+----------+------------------+------------------+------------------+-----------------+

+------------+----------+
|payment_type|trip_count|
+------------+----------+
|           1|    695347|
|           2|    121634|
|           4|     10858|
|           3|      3722|
+------------+----------+



In [0]:
output_path = "/Volumes/trip/trip/trip/cleaned_taxi_data"

df_clean = df   

df_clean.coalesce(1) \
    .write.mode("overwrite") \
    .option("header", "true") \
    .csv(output_path)

print("Saved successfully")

Saved successfully


2. Exploratory Data Analysis (EDA)

In [0]:
display(df.select("trip_distance", "fare_amount"))


trip_distance,fare_amount
0.2,4.4
0.2,3.7
0.2,5.1
0.3,5.8
0.3,6.5
0.3,6.5
0.3,5.1
0.3,4.4
0.3,5.1
0.4,5.1


Databricks visualization. Run in Databricks to view.

In [0]:
display(df.select("total_amount"))

total_amount
10.9
10.7
14.85
9.8
10.5
16.5
12.35
11.99
13.15
11.82


Databricks visualization. Run in Databricks to view.

In [0]:
display(df.groupBy("pickup_hour").avg("total_amount"))

pickup_hour,avg(total_amount)
18,20.395504898331556
12,18.89953526011558
10,18.78074965751688
5,18.21355797994017
3,19.275462530881132
1,19.82839925546759
22,20.141954739637487
14,18.945933878594758
17,20.963964194373325
11,18.499861224991776


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    df.groupBy("payment_type")
      .count()
      .orderBy("count", ascending=False)
)

payment_type,count
1,695347
2,121634
4,10858
3,3722


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import avg

display(
    df.groupBy("passenger_count")
      .agg(avg("total_amount").alias("avg_total"))
)

passenger_count,avg_total
5,19.6383776734481
1,19.441718330123322
3,20.129940456928367
2,19.94351490203077
6,19.538780957622453
4,20.52014017603501


Databricks visualization. Run in Databricks to view.

In [0]:
display(df.select("trip_duration"))

trip_duration
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0


Databricks visualization. Run in Databricks to view.

In [0]:
display(df.select("trip_distance", "total_amount"))

trip_distance,total_amount
0.2,10.9
0.2,10.7
0.2,14.85
0.3,9.8
0.3,10.5
0.3,16.5
0.3,12.35
0.3,11.99
0.3,13.15
0.4,11.82


Databricks visualization. Run in Databricks to view.

3. Feature Analysis


In [0]:
# Full summary statistics for key numeric columns
numeric_cols_eda = [
    "passenger_count", "trip_distance", "fare_amount", "extra",
    "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge",
    "total_amount", "congestion_surcharge", "Airport_fee", "trip_duration"
]

print("Full numeric summary statistics (count, mean, stddev, min, quartiles, max):")
display(
    df.select(*numeric_cols_eda)
      .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
)


Full numeric summary statistics (count, mean, stddev, min, quartiles, max):


summary,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,trip_duration
count,831561,831561,831561,831561,831561,831561,831561,831561,831561,831561,831561,831561
mean,1.349707357608161,1.684330061174109,12.058040757082308,1.2962391814911953,0.49955024345778604,2.4930207044342017,0.007961207896955265,0.9941110754352357,19.571393499694963,2.417769111346011,0.012273302860523761,10.95529973146963
stddev,0.7918735082045266,1.069998902751651,5.197165616028458,1.5174092684092968,0.014989204793646339,1.7140360343424956,0.23970354762889973,0.07651308497446889,6.158243904022721,0.44588736433463666,0.14603996607286682,41.30213974813939
min,1,0.01,0.01,0.0,0.0,0.0,0.0,0,1.76,0.0,0.0,0.0
25%,1,0.9,7.9,0.0,0.5,1.0,0.0,1,14.95,2.5,0.0,0.0
50%,1,1.4,10.7,1.0,0.5,2.61,0.0,1,18.5,2.5,0.0,0.0
75%,1,2.19,14.9,2.5,0.5,3.65,0.0,1,23.4,2.5,0.0,0.0
max,6,6.26,28.9,10.0,0.5,7.6,20.82,1,37.19,2.5,1.75,166.66666666666666


In [0]:
from pyspark.sql import functions as F

# Pairwise Pearson correlations for important numerical features
corr_cols = [
    "trip_distance", "fare_amount", "tip_amount", "total_amount",
    "passenger_count", "trip_duration", "pickup_hour"
]

corr_rows = []
for i, c1 in enumerate(corr_cols):
    for c2 in corr_cols[i + 1:]:
        corr_val = df.stat.corr(c1, c2)
        corr_rows.append((c1, c2, float(corr_val) if corr_val is not None else None))

corr_df = spark.createDataFrame(corr_rows, ["feature_1", "feature_2", "pearson_corr"])

print("Feature correlation pairs sorted by absolute correlation:")
display(corr_df.orderBy(F.abs(F.col("pearson_corr")).desc()))


Feature correlation pairs sorted by absolute correlation:


feature_1,feature_2,pearson_corr
fare_amount,total_amount,0.9360377206021606
trip_distance,fare_amount,0.889428120289029
trip_distance,total_amount,0.8379400109356704
tip_amount,total_amount,0.5952385742894477
fare_amount,tip_amount,0.33965067222687095
trip_distance,tip_amount,0.30384671757659176
fare_amount,trip_duration,0.13411281825142737
total_amount,trip_duration,0.12365020368426982
trip_distance,trip_duration,0.10446011873288716
total_amount,pickup_hour,0.07117568455788414


4. Machine Learning Model Implementation


In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

feature_cols = ["trip_distance", "passenger_count", "pickup_hour", "trip_duration"]
label_col = "total_amount"

# Build ML base table first (no scaling yet)
df_ml_base = df.dropna(subset=feature_cols + [label_col]).select(*feature_cols, label_col)

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_unscaled")
df_ml_base = assembler.transform(df_ml_base).select("features_unscaled", label_col)

# Proper split for model development
a_train_df, a_val_df, a_test_df = df_ml_base.randomSplit([0.7, 0.15, 0.15], seed=42)

print("Split sizes:")
print("Train rows:", a_train_df.count())
print("Validation rows:", a_val_df.count())
print("Test rows:", a_test_df.count())


Split sizes:
Train rows: 582242
Validation rows: 124519
Test rows: 124800


In [0]:
# Fit scaler ONLY on training data (prevents data leakage)
scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="scaled_features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(a_train_df)

train_df = scaler_model.transform(a_train_df).select("scaled_features", label_col)
val_df = scaler_model.transform(a_val_df).select("scaled_features", label_col)
test_df = scaler_model.transform(a_test_df).select("scaled_features", label_col)


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_eval = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
r2_eval = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="r2")
mae_eval = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="mae")


def metrics(pred_df):
    return (
        rmse_eval.evaluate(pred_df),
        r2_eval.evaluate(pred_df),
        mae_eval.evaluate(pred_df)
    )


In [0]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="scaled_features", labelCol=label_col)
lr_model = lr.fit(train_df)
lr_val_predictions = lr_model.transform(val_df)

lr_rmse, lr_r2, lr_mae = metrics(lr_val_predictions)
print("Linear Regression (Validation)")
print("RMSE:", lr_rmse)
print("R2:", lr_r2)
print("MAE:", lr_mae)


Linear Regression (Validation)
RMSE: 3.3000497234465778
R2: 0.7139102096355301
MAE: 2.4942641503090566


In [0]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(featuresCol="scaled_features", labelCol=label_col, seed=42)
dt_model = dt.fit(train_df)
dt_val_predictions = dt_model.transform(val_df)

dt_rmse, dt_r2, dt_mae = metrics(dt_val_predictions)
print("Decision Tree (Validation)")
print("RMSE:", dt_rmse)
print("R2:", dt_r2)
print("MAE:", dt_mae)


Decision Tree (Validation)
RMSE: 3.1626656005967275
R2: 0.7372347429812217
MAE: 2.3835766957962266


5. Optimisation & Tuning for Random Forest.

In [0]:
from pyspark.ml.regression import RandomForestRegressor

best_rf_rmse = float("inf")
best_rf_model = None
best_rf_params = None
best_rf_r2 = None
best_rf_mae = None

for trees in [50, 100]:
    for depth in [5, 10, 15]:
        rf = RandomForestRegressor(
            featuresCol="scaled_features",
            labelCol=label_col,
            numTrees=trees,
            maxDepth=depth,
            seed=42
        )
        rf_model = rf.fit(train_df)
        rf_val_predictions = rf_model.transform(val_df)

        rmse, r2, mae = metrics(rf_val_predictions)
        print(f"RF trees={trees}, depth={depth} -> RMSE={rmse}, R2={r2}, MAE={mae}")

        if rmse < best_rf_rmse:
            best_rf_rmse = rmse
            best_rf_model = rf_model
            best_rf_params = (trees, depth)
            best_rf_r2 = r2
            best_rf_mae = mae

print("Best RF params from validation:", best_rf_params)
print("Best RF validation RMSE:", best_rf_rmse)


RF trees=50, depth=5 -> RMSE=3.3116150613931583, R2=0.7119014380973854, MAE=2.5475066455546824
RF trees=50, depth=10 -> RMSE=3.1294778781956487, R2=0.7427205112512635, MAE=2.3740071886050425
RF trees=50, depth=15 -> RMSE=3.124970344712929, R2=0.7434611208819455, MAE=2.3695240469978405
RF trees=100, depth=5 -> RMSE=3.268122195923383, R2=0.719419189587247, MAE=2.501321569505606
RF trees=100, depth=10 -> RMSE=3.09196473625489, R2=0.7488515764164554, MAE=2.3316101565166423
RF trees=100, depth=15 -> RMSE=3.087762191018169, R2=0.7495338256773016, MAE=2.327683597052892
Best RF params from validation: (100, 15)
Best RF validation RMSE: 3.087762191018169


In [0]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="scaled_features",
    labelCol=label_col,
    maxIter=50,
    maxDepth=5,
    seed=42
)

gbt_model = gbt.fit(train_df)
gbt_val_predictions = gbt_model.transform(val_df)

gbt_rmse, gbt_r2, gbt_mae = metrics(gbt_val_predictions)
print("Gradient Boosted Trees (Validation)")
print("RMSE:", gbt_rmse)
print("R2:", gbt_r2)
print("MAE:", gbt_mae)


Gradient Boosted Trees (Validation)
RMSE: 3.067230902013703
R2: 0.752853574139487
MAE: 2.3025835548130944


In [0]:
# Validation model comparison table
validation_results = [
    ("Linear Regression", lr_rmse, lr_r2, lr_mae),
    ("Decision Tree", dt_rmse, dt_r2, dt_mae),
    (f"Random Forest tuned {best_rf_params}", best_rf_rmse, best_rf_r2, best_rf_mae),
    ("Gradient Boosted Trees", gbt_rmse, gbt_r2, gbt_mae)
]

validation_results_df = spark.createDataFrame(validation_results, ["Model", "RMSE", "R2", "MAE"])
print("Validation results (used for model selection):")
display(validation_results_df.orderBy("RMSE"))


Validation results (used for model selection):


Model,RMSE,R2,MAE
Gradient Boosted Trees,3.067230902013703,0.752853574139487,2.3025835548130944
"Random Forest tuned (100, 15)",3.087762191018169,0.7495338256773016,2.327683597052892
Decision Tree,3.1626656005967275,0.7372347429812217,2.3835766957962266
Linear Regression,3.3000497234465778,0.7139102096355301,2.4942641503090566


In [0]:
# Final model selection using validation RMSE (test set remains untouched)
if best_rf_rmse <= gbt_rmse and best_rf_rmse <= dt_rmse and best_rf_rmse <= lr_rmse:
    selected_model_name = "RandomForestRegressor"
    selected_params = {"numTrees": best_rf_params[0], "maxDepth": best_rf_params[1]}
elif gbt_rmse <= dt_rmse and gbt_rmse <= lr_rmse:
    selected_model_name = "GBTRegressor"
    selected_params = {"maxIter": 50, "maxDepth": 5}
elif dt_rmse <= lr_rmse:
    selected_model_name = "DecisionTreeRegressor"
    selected_params = {}
else:
    selected_model_name = "LinearRegression"
    selected_params = {}

print("Selected model:", selected_model_name)
print("Selected params:", selected_params)


Selected model: GBTRegressor
Selected params: {'maxIter': 50, 'maxDepth': 5}


In [0]:
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor, RandomForestRegressor, GBTRegressor

# Retrain selected model on train + validation data
train_val_df = train_df.unionByName(val_df)

if selected_model_name == "RandomForestRegressor":
    final_estimator = RandomForestRegressor(
        featuresCol="scaled_features",
        labelCol=label_col,
        numTrees=selected_params["numTrees"],
        maxDepth=selected_params["maxDepth"],
        seed=42
    )
elif selected_model_name == "GBTRegressor":
    final_estimator = GBTRegressor(
        featuresCol="scaled_features",
        labelCol=label_col,
        maxIter=selected_params["maxIter"],
        maxDepth=selected_params["maxDepth"],
        seed=42
    )
elif selected_model_name == "DecisionTreeRegressor":
    final_estimator = DecisionTreeRegressor(
        featuresCol="scaled_features",
        labelCol=label_col,
        seed=42
    )
else:
    final_estimator = LinearRegression(
        featuresCol="scaled_features",
        labelCol=label_col
    )

final_model = final_estimator.fit(train_val_df)
final_test_predictions = final_model.transform(test_df)

final_rmse, final_r2, final_mae = metrics(final_test_predictions)

print("FINAL TEST RESULTS (unseen data, evaluated once)")
print("Model:", selected_model_name)
print("RMSE:", final_rmse)
print("R2:", final_r2)
print("MAE:", final_mae)


FINAL TEST RESULTS (unseen data, evaluated once)
Model: GBTRegressor
RMSE: 3.068369932160222
R2: 0.7506233509736083
MAE: 2.3058784273643553
